In [1]:
import pandas as pd

In [16]:
# Load assignments
assign_df = pd.read_csv("phase3_assignments.csv")

# 1. Assignment success rate
total_clusters = len(assign_df)
successful = assign_df["vehicle_id"].notna().sum()
success_rate = round(100 * successful / total_clusters, 2)

# 2. Breakdown by vehicle type
type_counts = assign_df["vehicle_type"].value_counts(dropna=True)
ev_used     = type_counts.get("EV", 0)
diesel_used = type_counts.get("DIESEL", 0)
petrol_used = type_counts.get("PETROL", 0)

# 3. Average pickup distance
avg_dist = assign_df["distance_to_pickup_m"].dropna().mean()
veh = pd.read_csv("vehicles.csv")
idle_ev = veh[
    (veh["Type"].str.strip().str.upper() == "EV") &
    (veh["Status"].str.strip().str.lower() == "idle") &
    (veh["Battery_Percentage"] >= 30)
]

# 4. Print metrics
print("\n📊 Assignment Evaluation Metrics")
print("──────────────────────────────────────")
print(f"Total Clusters        : {total_clusters}")
print(f"Successful Assignments: {successful}")
print(f"Success Rate          : {success_rate:.2f}%")
print(f"→ EV Assigned         : {ev_used}")
print(f"→ Diesel Assigned     : {diesel_used}")
print(f"→ Petrol Assigned     : {petrol_used}")
total_ev_available = len(idle_ev)
print(total_ev_available)
ev_utilization = (534 / total_ev_available) * 100
print(f"->EV-Utilization:{ev_utilization}")



📊 Assignment Evaluation Metrics
──────────────────────────────────────
Total Clusters        : 1749
Successful Assignments: 1375
Success Rate          : 78.62%
→ EV Assigned         : 534
→ Diesel Assigned     : 423
→ Petrol Assigned     : 418
710
->EV-Utilization:75.21126760563381


In [ ]:
# evaluate_co2_and_distance.py
# --------------------------------------------------
import pickle, gzip, math
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

# ------------- Inputs -----------------------------
ASSIGN_FILE   = "phase3_assignments.csv"   # EV‑first result
VEHICLES_FILE = "vehicles.csv"
RIDES_CSV     = "Taxi_Trips__2024-__20250611.csv"                # To pull trip distance (km) if available
PHASE2_PKL_GZ = "phase2_clusters.pkl.gz"   # To get trip list per cluster

# ------------- Emission factors -------------------
EF_KG_PER_KM = {"PETROL": 0.192, "DIESEL": 0.222}   # kg CO₂ per km

# ============ 0. Helpers ==========================
RAD = math.pi / 180
def haversine(lat1, lon1, lat2, lon2):
    """scalar haversine km"""
    lat1 *= RAD; lon1 *= RAD; lat2 *= RAD; lon2 *= RAD
    dlat, dlon = lat2-lat1, lon2-lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return 6371.0 * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))  # km

# ============ 1. Load data ========================
assign   = pd.read_csv(ASSIGN_FILE)
vehicles = pd.read_csv(VEHICLES_FILE).rename(columns={
    "Vehicle_ID":"vehicle_id","Type":"type",
    "Current_Latitude":"lat","Current_Longitude":"lon"
})
vehicles["type"] = vehicles["type"].str.upper()

# Trip distance lookup (if you have Trip Miles)
if {"Trip_Miles","Trip_ID"}.issubset(pd.read_csv(RIDES_CSV, nrows=0).columns):
    dist_lookup = (pd.read_csv(RIDES_CSV, usecols=["Trip_ID","Trip_Miles"])
                     .set_index("Trip_ID")["Trip_Miles"]*1.60934)   # miles→km
else:
    dist_lookup = pd.Series(dtype=float)

# Map cluster → first trip pickup coords for dead‑head distance
with gzip.open(PHASE2_PKL_GZ,"rb") as f:
    clusters = {c["cluster_id"]: c for c in pickle.load(f)}

rides_pickup = (
    pd.read_csv(RIDES_CSV,
        usecols=["Trip_ID","Pickup_Centroid_Latitude","Pickup_Centroid_Longitude"])
      .set_index("Trip_ID")
)

# ============ 2. Add trip_km + deadhead_km =========
rows = []
for _, r in tqdm(assign.iterrows(), total=len(assign)):
    cid = r.cluster_id
    veh = vehicles.loc[vehicles.vehicle_id==r.vehicle_id]
    if veh.empty:
        continue
    v_lat, v_lon = veh.iloc[0][["lat","lon"]]
    # first member's pickup
    first_trip = clusters[cid]["members"][0]["trip_id"]
    p_lat, p_lon = rides_pickup.loc[first_trip][["Pickup_Centroid_Latitude","Pickup_Centroid_Longitude"]]
    deadhead_km = haversine(v_lat, v_lon, p_lat, p_lon)

    trip_km = dist_lookup.get(first_trip, 0)   # 0 if unknown
    rows.append({
        "vehicle_type": r.vehicle_type,
        "deadhead_km": deadhead_km,
        "trip_km": trip_km
    })

eval_df = pd.DataFrame(rows)
eval_df["total_km"] = eval_df["deadhead_km"] + eval_df["trip_km"]
eval_df["co2_kg"]   = eval_df.apply(
        lambda x: EF_KG_PER_KM.get(x.vehicle_type,0)*x.total_km, axis=1)

# ============ 3. Metrics for current (EV‑priority) run =========
by_type = eval_df.groupby("vehicle_type").agg(
    assignments = ("total_km","size"),
    avg_pickup_km = ("deadhead_km","mean"),
    total_km      = ("total_km","sum"),
    total_co2_kg  = ("co2_kg","sum")
).round(2)

print("\n📊 Current EV‑priority run")
print(by_type)
print("Total CO₂ (kg):", round(eval_df.co2_kg.sum(),2))

# ============ 4. Build baseline (closest‑vehicle‑wins, fuel‑blind) =========
#  Simplest baseline: assume the exact SAME clusters but ignore fuel priority.
#  We re‑assign each cluster to the single nearest idle vehicle (any type).
vehicles["idle"] = True
baseline_rows = []
for _, r in tqdm(assign.iterrows(), total=len(assign)):
    cid = r.cluster_id
    first_trip = clusters[cid]["members"][0]["trip_id"]
    p_lat, p_lon = rides_pickup.loc[first_trip][["Pickup_Centroid_Latitude","Pickup_Centroid_Longitude"]]

    # Compute distance to ALL still‑idle vehicles
    vehicles["dist"] = vehicles.apply(
        lambda v: haversine(v.lat,v.lon,p_lat,p_lon) if v.idle else 1e9, axis=1)
    best = vehicles.loc[vehicles.dist.idxmin()]
    vehicles.at[best.name,"idle"] = False

    deadhead_km = best.dist
    trip_km     = dist_lookup.get(first_trip,0)
    baseline_rows.append({
        "vehicle_type": best.type,
        "deadhead_km": deadhead_km,
        "trip_km": trip_km
    })

base_df = pd.DataFrame(baseline_rows)
base_df["total_km"] = base_df.deadhead_km + base_df.trip_km
base_df["co2_kg"]   = base_df.apply(
        lambda x: EF_KG_PER_KM.get(x.vehicle_type,0)*x.total_km, axis=1)

baseline_co2 = base_df.co2_kg.sum()
current_co2  = eval_df.co2_kg.sum()
reduction_pct = 100*(baseline_co2 - current_co2)/baseline_co2

print("\n🌍 Baseline (no EV priority) total CO₂:", round(baseline_co2,2), "kg")
print("🌱 Current EV‑priority total CO₂ :", round(current_co2,2), "kg")
print(f"🟢 CO₂ reduced by prioritising EVs: {reduction_pct:.2f}%")


100%|██████████| 1749/1749 [00:01<00:00, 1296.63it/s]



📊 Current EV‑priority run
              assignments  avg_pickup_km  total_km  total_co2_kg
vehicle_type                                                    
DIESEL                423           2.71   1147.72        254.79
EV                    534           2.74   1465.51          0.00
PETROL                418           2.95   1232.04        236.55
Total CO₂ (kg): 491.35


100%|██████████| 1749/1749 [00:43<00:00, 39.79it/s]


🌍 Baseline (no EV priority) total CO₂: 1204.21 kg
🌱 Current EV‑priority total CO₂ : 491.35 kg
🟢 CO₂ reduced by prioritising EVs: 59.20%
